<h2>使用 Selenium 自动化抓取</h2>

JavaScript 动态渲染的页面不止 Ajax 这一种。比如中国青年网,  https://news.youth.cn/gn/ 它的分页部分是由 JavaScript 生成的，并非原始 HTML 代码，这其中并不包含 Ajax 请求。

它即使是 Ajax 获取的数据，但是其 Ajax 接口含有很多加密参数，难以直接找出其规律，也很难直接分析 Ajax 来抓取。

为了解决这些问题，可以**直接使用模拟浏览器运行的方式来实现**，这样就可以做到在浏览器中看到是什么样，抓取的源码就是什么样，也就是可见即可爬。

Python 提供了许多模拟浏览器运行的库，如 `Selenium`、`Splash`、`PyV8`、`Ghost` 等。

<h3>Selenium 的使用</h3>

`Selenium` 是一个自动化测试工具，利用它可以驱动浏览器执行特定的动作，如点击、下拉等操作，同时还可以获取浏览器当前呈现的页面的源代码，做到可见即可爬。

以 Chrome 为例来讲解 `Selenium` 的用法。
> 在开始之前，确保已经正确安装好了 Chrome 浏览器并配置好了 ChromeDriver；还需要正确安装好 Python 的 `Selenium` 库。
>
> 运行代码后发现，会自动弹出一个 Chrome 浏览器。浏览器首先会跳转到百度，然后在搜索框中输入 Python，接着跳转到搜索结果页。

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait

browser = webdriver.Chrome()
try:
    browser.get('https://www.baidu.com')
    # 在Selenium 4之后的版本中，find_element_by_id() 方法已被弃用
    input = browser.find_element(By.ID, 'kw')
    input.send_keys('Python')
    input.send_keys(Keys.ENTER)
    wait = WebDriverWait(browser, 10)
    wait.until(EC.presence_of_element_located((By.ID, 'content_left')))
    print(browser.current_url)
    print(browser.get_cookies())
    # print(browser.page_source)
finally:
    browser.close()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


https://www.baidu.com/s?ie=utf-8&f=8&rsv_bp=1&rsv_idx=1&tn=baidu&wd=Python&fenlei=256&rsv_pq=0xd52aaef1011341be&rsv_t=0e31LNsIyW4MhbvuzLoqFrDo%2FUwvBmNKE1u7Wrlm8CWGZadARmAZmrnwiLbS&rqlang=en&rsv_enter=1&rsv_dl=tb&rsv_sug3=6&rsv_sug2=0&rsv_btype=i&inputT=56&rsv_sug4=56
[{'domain': 'www.baidu.com', 'expiry': 1754871265, 'httpOnly': False, 'name': 'COOKIE_SESSION', 'path': '/', 'sameSite': 'Lax', 'secure': False, 'value': '0_0_1_1_0_1_0_0_1_1_5_0_0_0_0_0_0_0_1723335271%7C1%230_0_1723335271%7C1'}, {'domain': '.baidu.com', 'httpOnly': False, 'name': 'delPer', 'path': '/', 'sameSite': 'Lax', 'secure': False, 'value': '0'}, {'domain': '.baidu.com', 'expiry': 1754871265, 'httpOnly': False, 'name': 'H_PS_PSSID', 'path': '/', 'sameSite': 'Lax', 'secure': False, 'value': '60272_60564_60571_60576'}, {'domain': '.baidu.com', 'httpOnly': False, 'name': 'PSINO', 'path': '/', 'sameSite': 'Lax', 'secure': False, 'value': '5'}, {'domain': '.baidu.com', 'expiry': 1723421665, 'httpOnly': False, 'name': 'B

**1. 声明浏览器对象（初始化）**

Selenium 支持非常多的浏览器，如 Chrome、Firefox、Edge 等，还有 Android、BlackBerry 等手机端的浏览器。另外，也支持无界面浏览器 PhantomJS。

```python
from selenium import webdriver

browser = webdriver.Chrome()
browser = webdriver.Firefox()
browser = webdriver.Edge()
browser = webdriver.PhantomJS()
browser = webdriver.Safari()
```

这样就完成了浏览器对象的初始化并将其赋值为 browser 对象。

**2. 调用 browser 对象，让其执行各个动作以模拟浏览器操作**

可以用 `get()` 方法来请求网页，参数传入链接 URL 即可。

```python
browser.get('https://www.taobao.com')
print(browser.page_source)
browser.close()
```

**3. 查找节点**

比如，想要从淘宝页面中提取搜索框这个节点:

![](7-1.png)

> 它的 `id` 是 `q`，`name` 也是 `q`。此外，还有许多其他属性，此时我们就可以用多种方式获取它了。
> 
> 比如，`find_element(By.NAME, 'q')` 是根据 name 值获取，`find_element(By.ID, 'q')` 是根据 id 获取。另外，还有根据 XPath、CSS 选择器等获取的方式。

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By

browser = webdriver.Chrome()
browser.get('https://www.taobao.com')
input_first = browser.find_element(By.ID, 'q')
input_second = browser.find_element(By.CSS_SELECTOR, '#q')
input_third = browser.find_element(By.XPATH, '//*[@id="q"]')
print(input_first, input_second, input_third)
browser.close()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


<selenium.webdriver.remote.webelement.WebElement (session="f8c96b9a3ba72d7388c9a298ad06753e", element="f.F26AB72D7973395DEB852510B6122FE9.d.134888184E5D63BCD8C3EF6D25A219BF.e.8")> <selenium.webdriver.remote.webelement.WebElement (session="f8c96b9a3ba72d7388c9a298ad06753e", element="f.F26AB72D7973395DEB852510B6122FE9.d.134888184E5D63BCD8C3EF6D25A219BF.e.8")> <selenium.webdriver.remote.webelement.WebElement (session="f8c96b9a3ba72d7388c9a298ad06753e", element="f.F26AB72D7973395DEB852510B6122FE9.d.134888184E5D63BCD8C3EF6D25A219BF.e.8")>


**ps. 多个节点**

如果要查找所有满足条件的节点，需要用 `find_elements()` 这样的方法。

例如，要查找淘宝左侧导航条的所有条目：

![](7-2.png)

就可以这样来实现：

In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By

browser = webdriver.Chrome()
browser.get('https://www.taobao.com')
lis = browser.find_elements(By.CSS_SELECTOR, '.service-bd--B9l1TEHT li')
for li in lis:
    print(li.text)
browser.close()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


电脑办公文具
工业品商业定制
家电手机数码
家具家装家居
女装男装配饰
女鞋男鞋运动
汽车珠宝箱包
食品生鲜健康
母婴童装潮玩
美妆洗护宠物
娱乐图书鲜花


**ps. 获取Cookies**

- 先登录 taobao
- 使用 `get_cookies()`获取 cookie

```python
from selenium import webdriver

browser = webdriver.Chrome()
browser.get('https://www.taobao.com/?spm=a21n57.1.logo.1.5b0a523cXbEAho')

taobao_cookies = browser.get_cookies()

for cookie in taobao_cookies:
    print("%s -> %s" % (cookie['name'], cookie['value']))
```

**5. 节点交互**

`Selenium` 可以驱动浏览器来执行一些操作。

比较常见的用法有：输入文字时用 `send_keys` 方法，清空文字时用 `clear` 方法，点击按钮时用 `click` 方法。示例如下：

In [12]:
from selenium import webdriver
import time

browser = webdriver.Chrome()
browser.get('https://www.taobao.com')

# browser.add_cookie(taobao_cookie)
# browser.refresh()

input = browser.find_element(By.ID, 'q')
input.send_keys('iPhone')
time.sleep(1)
input.clear()
input.send_keys('iPad')

button = browser.find_element(By.CLASS_NAME, 'btn-search')
button.click()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


更多的操作可以参见官方文档的交互动作介绍：https://selenium-python.readthedocs.io/api.html#module-selenium.webdriver.remote.webelement 

**6. 动作链**

在上面的实例中，一些交互动作都是针对某个节点执行的。（输入框，输入文字和清空文字方法；按钮，点击方法）

还有**一些操作没有特定的执行对象**，比如**鼠标拖曳、键盘按键**等，这些动作用另一种方式来执行，那就是动作链。

比如，现在实现一个节点的拖曳操作，将某个节点从一处拖曳到另外一处，可以这样实现：

In [9]:
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By

browser = webdriver.Chrome()
url = 'http://www.runoob.com/try/try.php?filename=jqueryui-api-droppable'
browser.get(url)
browser.switch_to.frame('iframeResult')
source = browser.find_element(By.CSS_SELECTOR, '#draggable')
target = browser.find_element(By.CSS_SELECTOR, '#droppable')
actions = ActionChains(browser)
actions.drag_and_drop(source, target)
actions.perform()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


更多的动作链操作可以参考官方文档的动作链介绍：http://selenium-python.readthedocs.io/api.html#module-selenium.webdriver.common.action_chains

**7. 执行 JavaScript**

对于某些操作，Selenium API 并没有提供。比如，下拉进度条，它可以直接模拟运行 JavaScript，此时使用 `execute_script()` 方法即可实现，代码如下：

In [10]:
from selenium import webdriver

browser = webdriver.Chrome()
browser.get('https://www.zhihu.com/explore')
browser.execute_script('window.scrollTo(0, document.body.scrollHeight)')
browser.execute_script('alert("To Bottom")')

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


**8. 获取节点信息**

通过 `page_source` 属性可以获取网页的源代码，接着就可以使用解析库（如`re`、`Beautiful Soup`、`pyquery` 等）提取信息。

`Selenium` 已经提供了选择节点的方法，返回的是 `WebElement` 类型，那么它也有相关的方法和属性来直接提取节点信息，如属性、文本等。这样的话，**可以不用通过解析源代码来提取信息**。

In [20]:
from selenium import webdriver
from selenium.webdriver.common.by import By

browser = webdriver.Chrome()
url = 'https://www.zhihu.com/explore'
browser.get(url)
logo = browser.find_element(By.CLASS_NAME, 'AppHeader-inner')
# 1. 获取属性
print(logo.get_attribute('class'))

# 2. 获取文本
a = browser.find_element(By.CLASS_NAME, 'css-1u3u1p5')
print(a.text)

# 3. 获取 ID、位置、标签名、大小
div = browser.find_element(By.CLASS_NAME, 'css-1fox6hm')
print(div.id)
print(div.location)
print(div.tag_name)
print(div.size)

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


AppHeader-inner css-11p8nt5
如何看待樊振东最新微博？
f.D74A98EB27969EA970254AB1EB5D4AF2.d.6DD54698F26FFF6F2B05181CD038CB87.e.11
{'x': 550, 'y': 144}
div
{'height': 72, 'width': 442}


**9. 切换 Frame**

网页中有一种节点叫作 iframe，也就是子 Frame，相当于页面的子页面，它的结构和外部网页的结构完全一致。

`Selenium` 打开页面后，它默认是在父级 Frame 里面操作，而此时如果页面中还有子 Frame，它是不能获取到子 Frame 里面的节点的。

这时就需要使用 `switch_to.frame()` 方法来切换 Frame。示例如下：

In [22]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

browser = webdriver.Chrome()
url = 'https://www.runoob.com/try/try.php?filename=jqueryui-api-droppable'
browser.get(url)
browser.switch_to.frame('iframeResult')
try:
    logo = browser.find_element(By.CLASS_NAME, 'logo')
except NoSuchElementException:
    print('NO LOGO')
browser.switch_to.parent_frame()
logo = browser.find_element(By.CLASS_NAME, 'logo')
print(logo)

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


NO LOGO
<selenium.webdriver.remote.webelement.WebElement (session="d629d454f3b3ac701cba778f664c9f8d", element="f.E2C4D63FE4995A6E81243571DC1E6215.d.D1169F5F7174D3A62DA55C4D7A875334.e.24")>



**10. 延时等待**

在 `Selenium` 中，`get()` 方法会在网页框架加载结束后结束执行，

此时如果获取 `page_source`，可能并不是浏览器完全加载完成的页面，

如果某些页面有额外的 Ajax 请求，我们在网页源代码中也不一定能成功获取到。所以，这里需要延时等待一定时间，确保节点已经加载出来。

这里等待的方式有两种：一种是隐式等待，一种是显式等待。

**隐式等待**

当使用隐式等待执行测试的时候，如果 `Selenium` 没有在 DOM 中找到节点，将继续等待，超出设定时间后，则抛出找不到节点的异常。
```python
from selenium import webdriver

browser = webdriver.Chrome()
browser.implicitly_wait(10)  # 隐式等 10 s
browser.get('https://www.zhihu.com/explore')
input = browser.find_element_by_class_name('zu-top-add-question')
print(input)
```

**显式等待**

隐式等待的效果其实并没有那么好，因为我们只规定了一个固定时间，而页面的加载时间会受到网络条件的影响。

显式等待是一种更合适的方法，它指定要查找的节点，然后指定一个最长等待时间。

**如果在规定时间内加载出来了这个节点，就返回查找的节点；如果到了规定时间依然没有加载出该节点，则抛出超时异常**。示例如下：
```python
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

browser = webdriver.Chrome()
browser.get('https://www.taobao.com/')
wait = WebDriverWait(browser, 10)
input = wait.until(EC.presence_of_element_located((By.ID, 'q')))
button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '.btn-search')))
print(input, button)
```

> 更多详细的等待条件的参数及用法介绍可以参考官方文档：https://selenium-python.readthedocs.io/api.html#module-selenium.webdriver.support.expected_conditions

**11. 前进后退**

平常使用浏览器时都有前进和后退功能，`Selenium` 也可以完成这个操作： `back()` 和 `forward()` 方法。示例如下：

In [24]:
import time
from selenium import webdriver

browser = webdriver.Chrome()
browser.get('https://www.baidu.com/')
browser.get('https://www.taobao.com/')
browser.get('https://www.python.org/')
browser.back()
time.sleep(1)
browser.forward()
browser.close()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


**12. 选项卡管理**

在访问网页的时候，会开启一个个选项卡。在 `Selenium` 中，我们也可以对选项卡进行操作。示例如下：

In [26]:
import time
from selenium import webdriver

browser = webdriver.Chrome()
browser.get('https://www.baidu.com')
browser.execute_script('window.open()')
print(browser.window_handles)
browser.switch_to.window(browser.window_handles[1])
browser.get('https://www.taobao.com')
time.sleep(1)
browser.switch_to.window(browser.window_handles[0])
browser.get('https://python.org')
time.sleep(1)
browser.close()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


['ABD13863D59D4E54BEF7253EF3E4AF6E', '9546EC63703D2CA15114E64AFE82FB09']


**13. 异常处理**

在使用 `Selenium` 的过程中，难免会遇到一些异常，例如超时、节点未找到等错误，一旦出现此类错误，程序便不会继续运行了。这里我们可以使用 `try except` 语句来捕获各种异常。

In [27]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, NoSuchElementException

browser = webdriver.Chrome()
try:
    browser.get('https://www.baidu.com')
except TimeoutException:
    print('Time Out')
try:
    browser.find_element(By.ID, 'hello')
except NoSuchElementException:
    print('No Element')
finally:
    browser.close()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


No Element


关于更多的异常类，可以参考官方文档：https://selenium-python.readthedocs.io/api.html#module-selenium.common.exceptions

<h3>Splash安装和使用</h3>

参考博文即可：https://blog.csdn.net/qq_53582111/article/details/121649717

- 推荐先安装 Docker